# 01 Smoke test: train, predict and score end to end

A deliberately tiny run on TNG300-3 dark matter (2 short epochs, small network) to check that the training, prediction and scoring code all work before spending GPU time on a real run. The scores will be poor; that is expected.

Runtime: **L4 GPU** with **High-RAM**. Needs the `GITHUB_TOKEN` secret, as in notebook 00.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!pip -q install --force-reinstall --no-deps "git+https://{token}@github.com/skumagai3/deepvoid.git"
del token
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import yaml
config = {
    'name': 'smoke_test',
    'output': '/content/runs',
    'data': {'inputs': ['tng300-3/dm'], 'labels': 'tng300-3/mask',
             'split': {'fractions': [0.7, 0.15, 0.15], 'buffer': 16, 'axis': 0}},
    'model': {'depth': 3, 'base_filters': 8, 'batch_norm': True},
    'training': {'crop': 64, 'batch_size': 4, 'steps_per_epoch': 20, 'val_steps': 4,
                 'epochs': 2, 'learning_rate': 1e-3, 'class_weights': None,
                 'mixed_precision': True, 'seed': 12},
}
yaml.safe_dump(config, open('/content/smoke.yaml', 'w'))

## 1. Train (a couple of minutes)

In [ ]:
from deepvoid.train import run
out = run('/content/smoke.yaml')
print(open(out / 'log.csv').read())

## 2. Predict the whole box and score the test slab

In [ ]:
import json, keras, numpy as np
from deepvoid import Catalog
from deepvoid.evaluate import confusion_matrix, scores
from deepvoid.predict import predict_volume
from deepvoid.preprocess import Normalizer
from deepvoid.split import Region

cat = Catalog()
model = keras.models.load_model(out / 'best.keras')
norm = Normalizer(**json.load(open(out / 'normalization.json'))['tng300-3/dm'])
x = norm(cat.load('tng300-3/dm').data)
y = cat.load('tng300-3/mask').data

probs = predict_volume(lambda b: model.predict_on_batch(b), x, crop=128, margin=16, batch_size=4)
test = Region(**json.load(open(out / 'split.json'))['test'])
s = scores(confusion_matrix(y, probs.argmax(-1), mask=test.mask(y.shape[0])))
print({k: round(v, 3) for k, v in s.items() if k in ('mcc', 'void_f1', 'balanced_accuracy', 'accuracy')})

Paste the output of the last two cells (or any error) back to Claude.